In [1]:
import pandas as pd
clean_01_fund_master = pd.read_csv("clean_01_fund_master.csv")
clean_02_nav_history = pd.read_csv("clean_02_nav_history.csv")
clean_03_aum_by_fund_house = pd.read_csv("clean_03_aum_by_fund_house.csv")
clean_04_monthly_sip_inflows = pd.read_csv("clean_04_monthly_sip_inflows.csv")
clean_05_category_inflows = pd.read_csv("clean_05_category_inflows.csv")
clean_06_industry_folio_count = pd.read_csv("clean_06_industry_folio_count.csv")
clean_07_scheme_performance = pd.read_csv("clean_07_scheme_performance.csv")
clean_08_investor_transactions = pd.read_csv("clean_08_investor_transactions.csv")
clean_09_portfolio_holdings = pd.read_csv("clean_09_portfolio_holdings.csv")
clean_07_scheme_performance = pd.read_csv("clean_07_scheme_performance.csv")


In [3]:
clean_02_nav_history = clean_02_nav_history.sort_values(
    ['amfi_code','date']
)

clean_02_nav_history['daily_return'] = (
    clean_02_nav_history
    .groupby('amfi_code')['nav']
    .pct_change()
)

In [4]:
clean_02_nav_history.head()

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210


In [5]:
def calculate_cagr(group):

    start_nav = group.iloc[0]['nav']
    end_nav = group.iloc[-1]['nav']

    years = (
        pd.to_datetime(group.iloc[-1]['date']) -
        pd.to_datetime(group.iloc[0]['date'])
    ).days / 365

    return ((end_nav/start_nav)**(1/years)-1)*100


In [6]:
cagr = (
    clean_02_nav_history
    .groupby('amfi_code')
    .apply(calculate_cagr, include_groups=False)
    .reset_index(name='cagr_pct')
)

In [7]:
volatility = (
    clean_02_nav_history
    .groupby('amfi_code')['daily_return']
    .std()
    * (252**0.5)
    * 100
)


In [8]:
risk_free_rate = 0.065
sharpe = (
    (cagr['cagr_pct']/100 - risk_free_rate)
    /
    (volatility.values/100)
)

In [9]:
def max_drawdown(group):

    running_max = group['nav'].cummax()

    drawdown = (
        group['nav']/running_max
    ) - 1

    return drawdown.min()*100

In [10]:
mdd = (
    clean_02_nav_history
    .groupby('amfi_code')
    .apply(max_drawdown, include_groups=False)
)


In [13]:
clean_07_scheme_performance.sort_values(
    "return_3yr_pct",
    ascending=False
)

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
29,101207,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Small Cap,Regular,24.93,22.38,23.80,20.54,1.84,0.97,0.90,1.47,25.0,-23.61,41613,1.53,5,Very High
27,119095,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,Small Cap,Regular,21.97,20.98,22.62,20.47,0.51,1.00,0.84,1.40,25.0,-14.45,21545,1.38,4,Very High
17,118634,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Small Cap,Regular,21.30,20.15,21.88,19.35,0.80,1.03,0.81,1.14,25.0,-30.87,43630,1.53,4,Very High
39,149324,DSP Small Cap Fund - Regular - Growth,DSP Mutual Fund,Small Cap,Regular,20.20,20.08,20.61,19.39,0.69,0.98,0.80,1.23,25.0,-17.01,35124,1.52,4,Very High
21,120842,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,Mid Cap,Regular,17.12,18.23,17.75,16.32,1.91,1.00,0.96,1.27,19.0,-21.92,47469,1.56,4,High
12,120505,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,Mid Cap,Regular,14.02,18.08,17.55,17.19,0.89,1.00,0.95,1.45,19.0,-21.84,979,1.36,3,High
38,149323,DSP Midcap Fund - Regular - Growth,DSP Mutual Fund,Mid Cap,Regular,14.12,17.16,19.00,16.14,1.02,0.98,0.90,1.50,19.0,-26.99,37835,1.61,4,High
7,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,Mid Cap,Regular,15.43,16.58,17.69,15.63,0.95,0.91,0.87,1.44,19.0,-13.67,23185,1.38,5,High
